# 取消行为与需求趋势分析

**分析目标**：
1. 从多维度剖析取消率（hotel、lead_time_group、deposit_type、market_segment、customer_type、special_requests）
2. 分析月度取消率趋势
3. 分析非取消订单的真实入住需求趋势（按 hotel 区分）

> **注意**：`set_plot_style()` 已修复中文字体配置，旧的乱码图片已删除。请 **按顺序重新运行本 Notebook 中所有生成图表的单元格**，以确保新保存的图片中文正常显示。需要重新运行的图表单元格包括：第 3、4、5、6、7、8、9、10 步中的绘图 + `save_figure()` 调用。

In [1]:
# 确保从 notebooks 目录运行时也能正确导入 src
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"项目根目录: {PROJECT_ROOT}")

项目根目录: c:\Users\Lenovo\Desktop\酒店项目\pandas-hotel-booking-analysis


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import PROCESSED_DATA_PATH, FIGURE_DIR
from src.analysis import (
    load_cleaned_data,
    calculate_overall_cancel_rate,
    calculate_cancel_rate_by_group,
    calculate_monthly_cancel_trend,
    calculate_monthly_demand_trend,
    calculate_special_request_cancel_rate,
)
from src.visualization import set_plot_style, save_figure

set_plot_style()
os.makedirs(FIGURE_DIR, exist_ok=True)

print("导入完成。")

[字体配置] 已加载中文字体: Microsoft YaHei (C:/Windows/Fonts/msyh.ttc)
[字体配置] rcParams 已设置为: Microsoft YaHei
导入完成。


---
## 第 1 步：读取清洗后数据

In [3]:
df = load_cleaned_data(PROCESSED_DATA_PATH)
print(f"数据集大小: {df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"日期范围: {df['arrival_date'].min().date()} → {df['arrival_date'].max().date()}")
df.head(3)

数据集大小: 119,390 行 × 42 列
日期范围: 2015-07-01 → 2017-08-31


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,has_agent,arrival_date,total_nights,total_guests,is_family,lead_time_group,adr_level,season,room_match,is_valid_guest
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,2015-07-01,0,2,0,180天以上,异常/免费,夏季,1,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,2015-07-01,0,2,0,180天以上,异常/免费,夏季,1,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,2015-07-01,1,1,0,0-7天,低,夏季,0,1


---
## 第 2 步：整体取消率

In [4]:
total, canceled, non_canceled, overall_rate = calculate_overall_cancel_rate(df)
print(f"整体预订数: {total:,}")
print(f"取消订单数: {canceled:,}")
print(f"未取消订单数: {non_canceled:,}")
print(f"整体取消率: {overall_rate:.2%}")

整体预订数: 119,390
取消订单数: 44,224
未取消订单数: 75,166
整体取消率: 37.04%


---
## 第 3 步：按 hotel 维度分析取消率

In [5]:
hotel_cancel = calculate_cancel_rate_by_group(df, "hotel")
hotel_cancel

,hotel,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,City Hotel,79330,33102,46228,0.417270
1,Resort Hotel,40060,11122,28938,0.277634


In [6]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette("Set2", len(hotel_cancel))
bars = ax.bar(hotel_cancel["hotel"], hotel_cancel["cancel_rate"], color=colors)
ax.set_title("按 酒店类型 分组的取消率", fontsize=14, fontweight="bold")
ax.set_xlabel("酒店类型")
ax.set_ylabel("取消率（%）")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars, hotel_cancel["cancel_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{val:.1%}", ha="center", fontsize=11)
fig.tight_layout()
save_figure(fig, "02_cancel_rate_by_hotel.png")

图表已保存: outputs/figures\02_cancel_rate_by_hotel.png


---
## 第 4 步：按 lead_time_group 维度分析取消率

In [7]:
lt_cancel = calculate_cancel_rate_by_group(df, "lead_time_group")
lt_cancel

,lead_time_group,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,180天以上,24692,14077,10615,0.570104
1,91-180天,26439,11821,14618,0.447105
2,31-90天,29553,11141,18412,0.376984
3,8-30天,18960,5283,13677,0.278639
4,0-7天,19746,1902,17844,0.096323


In [8]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Set2", len(lt_cancel))
bars = ax.bar(lt_cancel["lead_time_group"], lt_cancel["cancel_rate"], color=colors)
ax.set_title("按 提前预订天数分组 的取消率", fontsize=14, fontweight="bold")
ax.set_xlabel("提前预订天数分组")
ax.set_ylabel("取消率（%）")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars, lt_cancel["cancel_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{val:.1%}", ha="center", fontsize=10)
fig.tight_layout()
save_figure(fig, "02_cancel_rate_by_lead_time_group.png")

图表已保存: outputs/figures\02_cancel_rate_by_lead_time_group.png


---
## 第 5 步：按 deposit_type 维度分析取消率

In [9]:
deposit_cancel = calculate_cancel_rate_by_group(df, "deposit_type")
deposit_cancel

,deposit_type,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,Non Refund,14587,14494,93,0.993624
1,No Deposit,104641,29694,74947,0.283770
2,Refundable,162,36,126,0.222222


In [10]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Set2", len(deposit_cancel))
bars = ax.bar(deposit_cancel["deposit_type"], deposit_cancel["cancel_rate"], color=colors)
ax.set_title("按 押金类型 分组的取消率", fontsize=14, fontweight="bold")
ax.set_xlabel("押金类型")
ax.set_ylabel("取消率（%）")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars, deposit_cancel["cancel_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{val:.1%}", ha="center", fontsize=11)
fig.tight_layout()
save_figure(fig, "02_cancel_rate_by_deposit_type.png")

图表已保存: outputs/figures\02_cancel_rate_by_deposit_type.png


---
## 第 6 步：按 market_segment 维度分析取消率

In [11]:
market_cancel = calculate_cancel_rate_by_group(df, "market_segment")
market_cancel

,market_segment,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,Undefined,2,2,0,1.000000
1,Groups,19811,12097,7714,0.610620
2,Online TA,56477,20739,35738,0.367211
3,Offline TA/TO,24219,8311,15908,0.343160
4,Aviation,237,52,185,0.219409
5,Corporate,5295,992,4303,0.187347
6,Direct,12606,1934,10672,0.153419
7,Complementary,743,97,646,0.130552


In [12]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Set2", len(market_cancel))
bars = ax.bar(market_cancel["market_segment"], market_cancel["cancel_rate"], color=colors)
ax.set_title("按 市场细分 分组的取消率", fontsize=14, fontweight="bold")
ax.set_xlabel("市场细分")
ax.set_ylabel("取消率（%）")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars, market_cancel["cancel_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{val:.1%}", ha="center", fontsize=10)
fig.tight_layout()
save_figure(fig, "02_cancel_rate_by_market_segment.png")

图表已保存: outputs/figures\02_cancel_rate_by_market_segment.png


---
## 第 7 步：按 customer_type 维度分析取消率

In [13]:
cust_cancel = calculate_cancel_rate_by_group(df, "customer_type")
cust_cancel

,customer_type,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,Transient,89613,36514,53099,0.407463
1,Contract,4076,1262,2814,0.309617
2,Transient-Party,25124,6389,18735,0.254299
3,Group,577,59,518,0.102253


In [14]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("Set2", len(cust_cancel))
bars = ax.bar(cust_cancel["customer_type"], cust_cancel["cancel_rate"], color=colors)
ax.set_title("按 客户类型 分组的取消率", fontsize=14, fontweight="bold")
ax.set_xlabel("客户类型")
ax.set_ylabel("取消率（%）")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars, cust_cancel["cancel_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
            f"{val:.1%}", ha="center", fontsize=11)
fig.tight_layout()
save_figure(fig, "02_cancel_rate_by_customer_type.png")

图表已保存: outputs/figures\02_cancel_rate_by_customer_type.png


---
## 第 8 步：total_of_special_requests 与取消率关系

In [15]:
special_cancel = calculate_special_request_cancel_rate(df)
special_cancel

,total_of_special_requests,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,0,70318,33556,36762,0.477204
1,2,12969,2866,10103,0.220989
2,1,33226,7318,25908,0.220249
3,3,2497,446,2051,0.178614
4,4,340,36,304,0.105882
5,5,40,2,38,0.050000


In [16]:
fig, ax = plt.subplots(figsize=(10, 5))
x = special_cancel["total_of_special_requests"].astype(int)
bars = ax.bar(x, special_cancel["cancel_rate"],
              color=sns.color_palette("Set2", len(special_cancel)))
ax.set_title("特殊需求数量与取消率的关系", fontsize=14, fontweight="bold")
ax.set_xlabel("特殊需求数量")
ax.set_ylabel("取消率（%）")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
for bar, val in zip(bars, special_cancel["cancel_rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", fontsize=9)
fig.tight_layout()
save_figure(fig, "02_cancel_rate_by_special_requests.png")

图表已保存: outputs/figures\02_cancel_rate_by_special_requests.png


---
## 第 9 步：月度预订量与取消率趋势

In [17]:
monthly_cancel = calculate_monthly_cancel_trend(df)
monthly_cancel.head(12)

,month,total_bookings,canceled_bookings,non_canceled_bookings,cancel_rate
0,2015-07,2776,1259,1517,0.453530
1,2015-08,3889,1598,2291,0.410903
2,2015-09,5114,2094,3020,0.409464
3,2015-10,4957,1732,3225,0.349405
4,2015-11,2340,486,1854,0.207692
5,2015-12,2920,973,1947,0.333219
6,2016-01,2248,557,1691,0.247776
7,2016-02,3891,1337,2554,0.343613
8,2016-03,4824,1477,3347,0.306177
9,2016-04,5428,2061,3367,0.379698


In [18]:
fig, ax1 = plt.subplots(figsize=(14, 6))

color_bar = sns.color_palette("Set2")[0]
color_line = sns.color_palette("Set2")[3]

bars = ax1.bar(monthly_cancel["month"], monthly_cancel["total_bookings"],
               color=color_bar, alpha=0.7, label="总预订量")
ax1.set_xlabel("月份")
ax1.set_ylabel("预订数量", color=color_bar)
ax1.tick_params(axis="y", labelcolor=color_bar)
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly_cancel["month"], monthly_cancel["cancel_rate"],
         color=color_line, marker="o", linewidth=2, label="取消率")
ax2.set_ylabel("取消率（%）", color=color_line)
ax2.tick_params(axis="y", labelcolor=color_line)
ax2.set_ylim(0, 1)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

ax1.set_title("月度预订量与取消率趋势", fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "02_monthly_cancel_trend.png")

图表已保存: outputs/figures\02_monthly_cancel_trend.png


---
## 第 10 步：非取消订单的月度真实需求趋势（按 hotel 区分）

In [19]:
demand = calculate_monthly_demand_trend(df)
demand.head(10)

,month,hotel,demand
0,2015-07,City Hotel,459
1,2015-07,Resort Hotel,1058
2,2015-08,City Hotel,1248
3,2015-08,Resort Hotel,1043
4,2015-09,City Hotel,1986
5,2015-09,Resort Hotel,1034
6,2015-10,City Hotel,2065
7,2015-10,Resort Hotel,1160
8,2015-11,City Hotel,934
9,2015-11,Resort Hotel,920


In [20]:
fig, ax = plt.subplots(figsize=(14, 6))
hotels = demand["hotel"].unique()
colors = sns.color_palette("Set2", len(hotels))

for hotel, color in zip(hotels, colors):
    subset = demand[demand["hotel"] == hotel]
    ax.plot(subset["month"], subset["demand"],
            marker="o", linewidth=2, color=color, label=hotel)

ax.set_title("非取消订单的月度真实需求趋势（按酒店类型）", fontsize=14, fontweight="bold")
ax.set_xlabel("月份")
ax.set_ylabel("入住需求量")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="酒店类型")
fig.tight_layout()
save_figure(fig, "02_monthly_demand_by_hotel.png")

图表已保存: outputs/figures\02_monthly_demand_by_hotel.png


---
## 小结

本 Notebook 从多维度分析了取消行为与需求趋势，主要发现如下：

1. **整体取消率**：整体取消率高达 37%，说明取消行为在酒店预订中非常普遍，需要重点关注。

2. **酒店类型差异**：City Hotel 的取消率明显高于 Resort Hotel，可能与城市酒店客户的预订灵活性更高有关。

3. **提前预订时间与取消率正相关**：提前预订时间越长，取消率越高。提前 180 天以上的订单取消率约 57.0%，而 0-7 天内的仅约 10%。

4. **押金类型影响显著**：“No Deposit”的取消率最低，而“Non Refund”的取消率反而最高，可能是因为不可退款的订单仍被系统标记为取消而非入住。

5. **市场细分差异**：Groups 和 Offline TA/TO 的取消率较高，Direct 和 Aviation 的取消率较低，反映了不同渠道客户的行为差异。

6. **特殊需求越多取消越低**：特殊需求数量与取消率呈明显负相关，提出特殊需求的客户通常入住意愿更强。

7. **季节性波动**：月度取消率和需求量均呈现明显的季节性波动，夏季需求旺季但取消率也相应升高。